<a href="https://colab.research.google.com/github/Manasi1195/Live-Staffing-Calculator/blob/main/staffing_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

def erlang_c(N, A):
    if N <= A: return 1.0
    inv = 1.0
    for k in range(1, N + 1):
        inv = 1.0 + inv * k / A
    eb = 1.0 / inv
    return N * eb / (N - A * (1 - eb))

def service_level(N, A, T, aht):
    return 1.0 - erlang_c(N, A) * math.exp(-(N - A) * T / aht)

def avg_speed_of_answer(N, A, aht):
    return erlang_c(N, A) * aht / (N - A)

def min_agents(A, sl_pct, T, aht):
    N = max(math.ceil(A) + 1, 1)
    while service_level(N, A, T, aht) < sl_pct / 100:
        N += 1
        if N > 999: break
    return N

BELL = [0.30,0.50,0.70,0.90,1.00,1.10,1.20,1.15,1.10,1.00,0.90,0.80,
        0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20]

style  = {'description_width': '140px'}
layout = widgets.Layout(width='420px')

w_aht    = widgets.IntSlider(value=240, min=30,  max=600,   step=10,   description='AHT (sec)',         style=style, layout=layout)
w_sl     = widgets.IntSlider(value=80,  min=50,  max=99,               description='SL Target %',       style=style, layout=layout)
w_sat    = widgets.IntSlider(value=20,  min=5,   max=60,               description='Answer time (sec)', style=style, layout=layout)
w_hours  = widgets.IntSlider(value=12,  min=4,   max=24,               description='Operating hours',   style=style, layout=layout)
w_shrink = widgets.FloatSlider(value=0.20, min=0.05, max=0.50, step=0.01, description='Shrinkage',      style=style, layout=layout, readout_format='.0%')
w_calls  = widgets.BoundedIntText(value=1200, min=1, max=100000,       description='Daily calls',       style=style, layout=widgets.Layout(width='300px'))
w_btn    = widgets.Button(description='Calculate', button_style='primary', layout=widgets.Layout(width='180px', height='38px'))
out      = widgets.Output()

header = widgets.HTML('<h2 style="font-family:system-ui">Contact Center Staffing Calculator</h2><p style="color:#666;font-family:system-ui">Erlang-C based agent requirement planner &mdash; adjust inputs and click Calculate</p>')

col1 = widgets.VBox([w_aht, w_sl, w_sat])
col2 = widgets.VBox([w_hours, w_shrink, w_calls])
ui   = widgets.VBox([header, widgets.HBox([col1, col2]), w_btn, out],
                    layout=widgets.Layout(padding='10px'))

def calculate(btn=None):
    aht_v = w_aht.value
    sl_v  = w_sl.value
    sat_v = w_sat.value
    hrs   = w_hours.value
    shrk  = w_shrink.value
    dc    = w_calls.value

    start      = (24 - hrs) // 2
    weights    = BELL[start: start + hrs]
    wsum       = sum(weights)
    calls_list = [round(dc * w / wsum) for w in weights]

    labels = []
    for i in range(hrs):
        h = start + i
        labels.append(f"{h % 12 or 12}{'AM' if h < 12 else 'PM'}")

    rows = []
    for i, calls in enumerate(calls_list):
        if calls == 0:
            rows.append({'Hour': labels[i], 'Calls': 0, 'Erlangs': 0.0,
                         'Min N': 0, 'Headcount': 0, 'Util %': 0.0,
                         'ASA (s)': 0.0, 'SL %': 100.0})
            continue
        A  = (calls / 3600) * aht_v
        N  = min_agents(A, sl_v, sat_v, aht_v)
        hc = math.ceil(N / (1 - shrk))
        rows.append({
            'Hour':      labels[i],
            'Calls':     calls,
            'Erlangs':   round(A, 2),
            'Min N':     N,
            'Headcount': hc,
            'Util %':    round(A / N * 100, 1),
            'ASA (s)':   round(avg_speed_of_answer(N, A, aht_v), 1),
            'SL %':      round(service_level(N, A, sat_v, aht_v) * 100, 1),
        })

    df       = pd.DataFrame(rows)
    active   = df[df['Calls'] > 0]
    peak_hc  = int(df['Headcount'].max())
    avg_sl   = active['SL %'].mean()
    avg_asa  = active['ASA (s)'].mean()
    avg_util = active['Util %'].mean()

    with out:
        clear_output(wait=True)

        fig = plt.figure(figsize=(16, 11))
        fig.patch.set_facecolor('#f5f6fa')
        gs  = gridspec.GridSpec(3, 4, figure=fig,
                                height_ratios=[0.8, 2.2, 2.2],
                                hspace=0.55, wspace=0.35)

        metric_data = [
            (str(peak_hc),       f'Peak headcount\n(+{shrk*100:.0f}% shrinkage)', '#185FA5'),
            (f'{avg_sl:.1f}%',   f'Avg service level\ntarget {sl_v}%',
             '#2E7D32' if avg_sl >= sl_v else '#C62828'),
            (f'{avg_asa:.0f}s',  'Avg speed of answer\n(ASA)', '#6A1B9A'),
            (f'{avg_util:.1f}%', 'Avg agent utilisation',
             '#E65100' if avg_util > 85 else '#00695C'),
        ]
        for i, (val, lbl, color) in enumerate(metric_data):
            ax = fig.add_subplot(gs[0, i])
            ax.set_facecolor('white')
            for sp in ax.spines.values(): sp.set_color('#dde1ea')
            ax.set_xticks([]); ax.set_yticks([])
            ax.text(0.5, 0.58, val, ha='center', va='center',
                    fontsize=28, fontweight='bold', color=color, transform=ax.transAxes)
            ax.text(0.5, 0.18, lbl, ha='center', va='center',
                    fontsize=9.5, color='#555', transform=ax.transAxes, linespacing=1.5)

        x = list(range(len(labels)))

        ax1 = fig.add_subplot(gs[1, :2])
        ax1.set_facecolor('white')
        ax1.bar([i - 0.2 for i in x], df['Headcount'], width=0.38,
                color='#378ADD', label='Headcount (rostered)', zorder=2)
        ax1.bar([i + 0.2 for i in x], df['Min N'], width=0.38,
                color='#9FE1CB', label='Min agents (Erlang-C)', zorder=2)
        ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
        ax1.set_title('Intraday Staffing Requirements', fontweight='bold', fontsize=11)
        ax1.set_ylabel('Agents'); ax1.legend(fontsize=9)
        ax1.grid(axis='y', alpha=0.3, zorder=0)
        for sp in ax1.spines.values(): sp.set_color('#dde1ea')

        ax2  = fig.add_subplot(gs[1, 2:])
        ax2b = ax2.twinx()
        ax2.set_facecolor('white')
        ax2.bar(x, df['Calls'], color='#E8ECF5', label='Calls', zorder=2)
        ax2b.plot(x, df['SL %'], color='#185FA5', marker='o',
                  markersize=5, linewidth=2, label='SL %', zorder=3)
        ax2b.axhline(sl_v, color='#E05A5A', linestyle='--',
                     linewidth=1.5, label=f'Target ({sl_v}%)', zorder=3)
        ax2b.set_ylim(0, 115); ax2b.set_ylabel('Service Level %')
        ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
        ax2.set_title('Call Volume vs Service Level', fontweight='bold', fontsize=11)
        ax2.set_ylabel('Call Volume')
        ax2.grid(axis='y', alpha=0.3, zorder=0)
        for sp in ax2.spines.values(): sp.set_color('#dde1ea')
        l1, lb1 = ax2.get_legend_handles_labels()
        l2, lb2 = ax2b.get_legend_handles_labels()
        ax2.legend(l1 + l2, lb1 + lb2, fontsize=9, loc='upper left')

        ax3 = fig.add_subplot(gs[2, :])
        ax3.axis('off')
        cols  = ['Hour', 'Calls', 'Erlangs', 'Min N', 'Headcount', 'Util %', 'ASA (s)', 'SL %']
        tdata = [[
            r['Hour'], r['Calls'], f"{r['Erlangs']:.2f}", r['Min N'],
            r['Headcount'], f"{r['Util %']:.1f}", f"{r['ASA (s)']:.1f}", f"{r['SL %']:.1f}"
        ] for _, r in df.iterrows()]

        tbl = ax3.table(cellText=tdata, colLabels=cols, loc='center', cellLoc='center')
        tbl.auto_set_font_size(False); tbl.set_fontsize(9.5)
        tbl.auto_set_column_width(list(range(len(cols))))
        tbl.scale(1, 1.6)

        for (r, c), cell in tbl.get_celld().items():
            cell.set_linewidth(0.5)
            if r == 0:
                cell.set_facecolor('#185FA5')
                cell.set_text_props(color='white', fontweight='bold')
            elif r % 2 == 0:
                cell.set_facecolor('#f0f4fb')
            else:
                cell.set_facecolor('white')
            if c == 7 and r > 0:
                sl_val = float(df['SL %'].iloc[r - 1])
                cell.set_facecolor('#d4edda' if sl_val >= sl_v else '#f8d7da')
            if c == 5 and r > 0:
                u_val = float(df['Util %'].iloc[r - 1])
                if u_val > 85: cell.set_facecolor('#fff3cd')

        ax3.set_title('Intraday Staffing Breakdown', fontweight='bold', fontsize=11, pad=12, loc='left')

        fig.suptitle('Contact Center Staffing Dashboard',
                     fontsize=15, fontweight='bold', color='#1a1a2e', y=1.005)
        plt.savefig('staffing_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#f5f6fa')
        plt.show()

w_btn.on_click(calculate)
display(ui)
calculate()
